# P4 — paired background-flattening control (exploratory)

选择 **T4 GPU → 全部运行**。默认同时跑两个模型，自动完成 smoke、full 和汇总。
这次会做新的小规模推理；只读 notebook 14 已完成，不用再跑。
保留原 32 个 A IDs/领域、三个 seeds、两种机制和全部八个 cells。

两个条件在同一运行环境中配对：

- **original_background:** 重新运行原目标历史 `b(t) + r(t)`；
- **mean_background:** 只替换为 `mean(b_history) + r(t)`。

`r(t)` 的原幅值尺度不重新估计；covariates、pulse、oracle、模型 revision 不变。
不读取真实未来 target，不运行 target-only/SQL，不访问 B，不改原判定或论文。
它同时改变背景时间变化、目标历史总方差/频谱，不能独立识别某一种模型机制。
控制条件改善也不等于真实背景迁移成功。

需要 Drive 上原 electricity/tourism discovery 输出、`p4_*_runtime_*` 输入缓存，
以及 notebook 14 的 `p4_response_diagnostics_v1/response_diagnostics_report.json`。
自动定位并核对原 release；**不下载新领域数据，不重新选 IDs**。
缺失时会停止并提示缺哪个原缓存，不会用替代数据继续。

断线后保持参数，重新全部运行即可；已校验单元会恢复，所有原文件保持不变。
新输出：`MyDrive/tsfm-covariate-faithfulness/p4_background_control_v1/`。
最终下载汇总 ZIP，不包含 NPZ、原始 IDs 或输入背景。
新指标和配对差值仅作描述性诊断，无新 CI、p 值或 complete-pattern 判定。


In [ ]:
BACKBONE = 'both'  # both / chronos_2 / timesfm_3
MODE = 'full'  # full automatically runs/checks smoke; smoke or analyze also available
assert BACKBONE in {'both', 'chronos_2', 'timesfm_3'}
assert MODE in {'full', 'smoke', 'analyze'}
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness')
OUTPUT = PROJECT / 'p4_background_control_v1'
print('Exploratory paired control:', MODE, BACKBONE)


In [ ]:
EXPECTED_SOURCE_SHA256 = '7bb0fd13a5a9015da993c8fa6d08bcab4c4bf4ca93597b91afb0140de83529c6'
EXPECTED_CONFIG_HASH = '12be6fd2aea32a5b3773362f36e2da5aab3d07dbcc66bae43a16b9650acf00aa'
CODE_ZIP_BASE64 = '...'
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import zipfile

blob = base64.b64decode(CODE_ZIP_BASE64, validate=True)
assert hashlib.sha256(blob).hexdigest() == EXPECTED_SOURCE_SHA256
BOOT = Path('/content') / ('p4_background_control_code_' + EXPECTED_SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names))
    for name in names:
        path = (BOOT / name).resolve()
        assert path.is_relative_to(BOOT.resolve()), 'Unsafe code member'
        content = archive.read(name)
        if path.exists():
            assert path.read_bytes() == content, f'Existing snapshot differs: {name}'
        else:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(content)
    assert {p.relative_to(BOOT).as_posix() for p in BOOT.rglob('*') if p.is_file()} == set(names)
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(BOOT / 'src')
ENV['PYTHONDONTWRITEBYTECODE'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install',
                '-r', str(BOOT / 'requirements/colab-base.txt')], check=True)

def run_logged(mode, backbone=None):
    command = [sys.executable, '-u', '-m', 'covfaith_p4.background_control',
               '--repo', str(BOOT), '--project', str(PROJECT), '--output', str(OUTPUT),
               '--mode', mode]
    if backbone is not None:
        command += ['--backbone', backbone]
    # Logs are separate from immutable input/output artifacts.
    log_path = OUTPUT / 'logs' / f'{backbone or "cpu"}_{mode}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        with subprocess.Popen(command, cwd=BOOT, env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Stopped; copy the actual error above. Keep old files. Log: {log_path}')
print('Code/config snapshot:', EXPECTED_SOURCE_SHA256, EXPECTED_CONFIG_HASH)


In [ ]:
# CPU preparation verifies both complete original archives before any model load.
run_logged('prepare')
preparation = json.loads((OUTPUT / 'preparation.json').read_text())
assert preparation['config_hash'] == EXPECTED_CONFIG_HASH
print('Verified original contexts and oracle; paired inputs durably saved.')


In [ ]:
backbones = ['chronos_2', 'timesfm_3'] if BACKBONE == 'both' else [BACKBONE]
if MODE != 'analyze':
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print('HF_TOKEN unavailable; public checkpoint download remains allowed.')
    else:
        ENV['HF_TOKEN'] = hf_token
        del hf_token
    for backbone in backbones:
        if backbone == 'timesfm_3':
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'timesfm'], check=True)
        requirement = 'chronos2.txt' if backbone == 'chronos_2' else 'timesfm3.txt'
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        '-r', str(BOOT / 'requirements' / requirement)], check=True)
        gpu_check = ('import torch; assert torch.cuda.is_available(), '
                     '"Select T4 GPU and reconnect"; print(torch.cuda.get_device_name(0))')
        subprocess.run([sys.executable, '-c', gpu_check], env=ENV, check=True)
        if backbone == 'timesfm_3':
            subprocess.run([sys.executable, '-c',
                            'from timesfm3 import ModelConfig, TimesFM3Evaluator'],
                           env=ENV, check=True)
        run_logged(MODE, backbone)
ANALYSIS_RAN = MODE == 'analyze' or (MODE == 'full' and BACKBONE == 'both')
if ANALYSIS_RAN:
    run_logged('analyze')
else:
    print('Use MODE=full and BACKBONE=both to complete both backbones and analyze.')


In [ ]:
if ANALYSIS_RAN:
    report = json.loads((OUTPUT / 'analysis/background_control_report.json').read_text())
    print(json.dumps({k: report[k] for k in ('result_status', 'config_hash', 'counts', 'scope')},
                     indent=2))
    selected = ['rgr', 'full_l1_ratio', 'absolute_full_gain_error', 'absolute_rgr_error',
                'model_outside_support_fraction', 'centroid_abs_error_steps',
                'peak_abs_error_steps', 'dsa', 'd1', 'g']
    for key, value in report['cells'].items():
        print(json.dumps({'cell': key, 'arms_and_paired_delta': {
            arm: {metric: values[metric]['point'] for metric in selected}
            for arm, values in value['arms_and_paired_delta'].items()},
            'original_rerun_vs_archive': value['original_rerun_vs_archive']}, indent=2))
    print('No new intervals/gates. The paired control is NOT a successful domain replication.')


In [ ]:
SUMMARY_FILES = ['background_control_report.json',
                 'complete_eight_cell_background_control.csv',
                 'per_seed_background_control.csv', 'artifact_manifest.json']
if ANALYSIS_RAN:
    from google.colab import files
    manifest = json.loads((OUTPUT / 'analysis/artifact_manifest.json').read_text())
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for name in SUMMARY_FILES:
            content = (OUTPUT / 'analysis' / name).read_bytes()
            if name != 'artifact_manifest.json':
                assert hashlib.sha256(content).hexdigest() == manifest[name], name
            info = zipfile.ZipInfo(name, date_time=(2026, 9, 30, 0, 0, 0))
            info.compress_type = zipfile.ZIP_DEFLATED
            archive.writestr(info, content)
    export = PROJECT / 'p4_background_control_summary_v1.zip'
    if export.exists():
        assert export.read_bytes() == buffer.getvalue(), 'Existing summary differs; preserve it.'
    else:
        export.write_bytes(buffer.getvalue())
    print('Send this summary ZIP only:', export)
    files.download(str(export))
